# Análisis Exploratorio de Datos (EDA) - Paysim (Detección de Fraude)

Este notebook tiene como objetivo realizar un análisis visual y estadístico del dataset para comprender mejor el comportamiento de las transacciones normales frente a las fraudulentas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="pastel")

print("Cargando dataset... (Esto puede tomar un minuto)")
df = pd.read_csv('PS_20174392719_1491204439457_log.csv')
display(df.head())

### 1. Información General del Dataset

In [ ]:
print("Forma del dataset:", df.shape)
print("\n--- Información de las columnas ---")
df.info()

### 2. Desbalance de la Clase Objetivo (`isFraud`)

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_perc = df['isFraud'].value_counts(normalize=True) * 100

print("Distribución de clases:")
print(fraud_counts)
print("\nPorcentaje de clases:")
print(fraud_perc)

plt.figure(figsize=(8, 5))
ax = sns.countplot(x='isFraud', data=df, palette=['#4CAF50', '#F44336'])
plt.title('Distribución de Transacciones: Normal (0) vs Fraude (1)', fontsize=14)
plt.yscale('log') # Escala logarítmica para poder ver la barra de fraude
plt.ylabel('Cantidad (Escala Logarítmica)')
plt.show()

Como podemos observar, el dataset está **profundamente desbalanceado**. Los fraudes representan menos del 0.13% del total, lo que justifica perfectamente el uso de modelos de Detección de Anomalías.

### 3. Tipos de Transacciones (`type`)

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='type', data=df, order=df['type'].value_counts().index, palette="viridis")
plt.title('Frecuencia de Tipos de Transacciones', fontsize=14)
plt.ylabel('Cantidad')
plt.show()

print("Fraudes por tipo de transacción:")
display(pd.crosstab(df['type'], df['isFraud']))

**Hallazgo Crucial:** ¡El fraude SOLO ocurre en transacciones de tipo **TRANSFER** y **CASH_OUT**! Esto significa que los estafadores transfieren dinero a una cuenta y luego lo retiran en efectivo. Ningún otro tipo de transacción presenta fraudes en este dataset.

### 4. Análisis de Cantidades Transadas (`amount`)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='isFraud', y='amount', data=df, palette=['#4CAF50', '#F44336'])
plt.title('Distribución de Montos (Amount) por Clase', fontsize=14)
plt.yscale('log') # Escala logarítmica para mejor visualización
plt.show()

print("Estadísticas de montos en transacciones NORMALES:")
display(df[df['isFraud'] == 0]['amount'].describe())

print("\nEstadísticas de montos en transacciones FRAUDULENTAS:")
display(df[df['isFraud'] == 1]['amount'].describe())

### 5. Correlación entre Variables Numéricas
Vamos a ver cómo se relacionan los montos transados con los balances de origen y destino.

In [ ]:
numerical_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud']
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title('Matriz de Correlación (Variables Numéricas)', fontsize=15)
plt.show()